# 📓 Text AI Module 4: ReAct (Reasoning + Acting) Agents & Tool Execution
Welcome to Module 4 of Language Models! In this notebook, we build an autonomous **ReAct Agent** (Yao et al., 2022) from scratch using a real open-weights Large Language Model (`Qwen/Qwen2.5-0.5B-Instruct` / `gpt2` fallback) [cite: 14].

---

## 💡 1. The ReAct (Reasoning + Acting) Paradigm
While standalone LLMs excel at text generation, they struggle with precise mathematical calculation, real-time web retrieval, and multi-step external API execution [cite: 14].

**ReAct** (Reasoning + Acting) overcomes these limitations by interleaving natural language reasoning with tool execution in an iterative loop [cite: 14]:

```
  User Problem -> Prompt + System Rules + Memory
                     │
                     ▼
         ┌──────────────────────┐
         │    Generate Thought   │  (LLM reasons about next step)
         └──────────┬───────────┘
                    │
                    ▼
         ┌──────────────────────┐
         │   Determine Action   │  (e.g., Action: calculator[25 * 3 * 5])
         └──────────┬───────────┘
                    │
                    ▼
         ┌──────────────────────┐
         │  Execute Tool Call   │  (Python function returns result)
         └──────────┬───────────┘
                    │
                    ▼
         ┌──────────────────────┐
         │ Receive Observation  │  (Appended to Memory)
         └──────────┬───────────┘
                    │
            [Is Task Complete?]
            ├── No  ──> Loop back to Generate Thought
            └── Yes ──> Output Final Answer
```

### Key ReAct Components:
1. **Thought:** The model's internal rationale outlining what information is needed next [cite: 14].
2. **Action:** The structured tool invocation string (e.g., `Action: calculator[expr]`) [cite: 14].
3. **Observation:** The environment/tool execution feedback appended back to the agent's memory [cite: 14].
4. **Final Answer:** The concluding response once all intermediate sub-problems are resolved [cite: 14].

In [ ]:
# Install required packages if running in Colab/Jupyter
try:
    import transformers
except ImportError:
    !pip install -q transformers torch accelerate ipywidgets matplotlib

import re
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from IPython.display import display, clear_output
import ipywidgets as widgets
from ipywidgets import interact, Dropdown, Text, IntSlider

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load small open-weights LLM (Qwen2.5-0.5B or Qwen3.5-0.8B)
MODEL_ID = "Qwen/Qwen3.5-0.8B"
# MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

print(f"Loading tokenizer and model: {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32).to(device)


model.eval()
print(f"Successfully loaded {MODEL_ID} on {device}!")

## 🛠️ 2. Defining Agent Tools & Sandbox Execution
We define custom Python functions that serve as external tools for our ReAct agent [cite: 14].

In [ ]:
def tool_calculator(expression: str) -> str:
    """Evaluates mathematical expressions safely."""
    try:
        sanitized = re.sub(r'[^0-9+\-*/().^]', '', expression).replace('^', '**')
        result = eval(sanitized)
        return f"{result}"
    except Exception as e:
        return f"Calculation Error: {e}"

def tool_unit_converter(query: str) -> str:
    """Converts basic units like km to miles or c to f."""
    try:
        query = query.lower().strip()
        if "km to miles" in query:
            val = float(query.split("km")[0])
            return f"{val * 0.621371:.2f} miles"
        elif "c to f" in query:
            val = float(query.split("c")[0])
            return f"{(val * 9/5) + 32:.2f} F"
        else:
            return "Conversion format not recognized. Supported: 'X km to miles', 'X C to F'."
    except Exception as e:
        return f"Conversion Error: {e}"

# Registry of available tools
AGENT_TOOLS = {
    "calculator": tool_calculator,
    "unit_converter": tool_unit_converter
}

print("Registered Tools:", list(AGENT_TOOLS.keys()))

## ⚙️ 3. ReAct System Prompt & Multi-Step Execution Class
We engineer a structured ReAct prompt template and build an agent runner that parses tool actions (`Action: tool_name[argument]`), executes them against `AGENT_TOOLS`, and appends `Observation:` results back into memory [cite: 14].

In [ ]:
REACT_SYSTEM_PROMPT = (
    "Solve the problem step by step using a Thought, Action, Observation loop.\n\n"
    "Available tools:\n"
    "- calculator[expression]: Evaluates a math expression (e.g., calculator[25 * 3 * 5]).\n"
    "- unit_converter[query]: Converts units (e.g., unit_converter[10 km to miles] or unit_converter[25 C to F]).\n\n"
    "Use the following exact format:\n"
    "Thought: <reasoning step>\n"
    "Action: <tool_name>[<argument>]\n\n"
    "When the answer is ready, output:\n"
    "Final Answer: <your final answer>\n"
)

class ReActAgent:
    def __init__(self, model, tokenizer, tools, max_steps=4):
        self.model = model
        self.tokenizer = tokenizer
        self.tools = tools
        self.max_steps = max_steps
        
    @torch.no_grad()
    def run(self, user_task):
        memory = f"{REACT_SYSTEM_PROMPT}\nQuestion: {user_task}\n"
        history_logs = []
        
        for step in range(1, self.max_steps + 1):
            inputs = self.tokenizer(memory, return_tensors="pt").to(device)
            
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=70,
                pad_token_id=self.tokenizer.eos_token_id,
                do_sample=False,  # Deterministic execution
                stop_strings=["Observation:"],  # Stop when expecting observation
                tokenizer=self.tokenizer
            )
            
            response = self.tokenizer.decode(output_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
            history_logs.append(f"Step {step} LLM Output:\n{response}")
            
            # Append response to memory
            memory += f"{response}\n"
            
            # Check if agent produced Final Answer
            if "Final Answer:" in response:
                final_ans = response.split("Final Answer:")[-1].strip()
                return final_ans, history_logs, memory
            
            # Parse Action: tool_name[arg]
            action_match = re.search(r"Action:\s*(\w+)\[(.*?)\]", response)
            if action_match:
                tool_name, tool_arg = action_match.groups()
                if tool_name in self.tools:
                    observation = self.tools[tool_name](tool_arg)
                else:
                    observation = f"Tool '{tool_name}' is not in available tools."
                
                obs_str = f"Observation: {observation}"
                history_logs.append(f"Step {step} Environment Observation: {obs_str}")
                memory += f"{obs_str}\n"
            else:
                # If model didn't format an Action or Final Answer, force step forward
                obs_str = "Observation: Please provide an Action in format tool_name[arg] or output Final Answer: <ans>"
                memory += f"{obs_str}\n"
                
        return "Reached maximum execution steps without Final Answer.", history_logs, memory

agent = ReActAgent(model, tokenizer, AGENT_TOOLS, max_steps=4)
print("ReAct Agent initialized successfully!")

## 🎛️ 4. Interactive ReAct Agent Playground
Type custom multi-step math or conversion tasks below to watch the ReAct agent reason, invoke tools, receive observations, and reach the final solution [cite: 14]!

In [ ]:
def interactive_agent_lab(task_query="Joe throws 25 punches per minute. How many punches in 5 rounds of 3 minutes?", max_steps=4):
    agent.max_steps = max_steps
    final_answer, logs, full_memory = agent.run(task_query)
    
    clear_output(wait=True)
    print(f"LLM Agent Engine : {MODEL_ID}")
    print(f"Task Query       : {task_query}")
    print("=" * 70)
    
    print("=== EXECUTION TRAJECTORY ===")
    for log in logs:
        print(log)
        print("-" * 50)
        
    print("=" * 70)
    print(f"RESULT -> Final Answer: {final_answer}")
    print("=" * 70)

interact(interactive_agent_lab,
         task_query=Text(value="Joe throws 25 punches per minute. How many punches in 5 rounds of 3 minutes?", description="Task:"),
         max_steps=IntSlider(value=4, min=1, max=8, step=1, description="Max Steps:"));